In [ ]:
from typing import Dict

import lftk
import pandas as pd
import spacy
import joblib

from catboost import CatBoostClassifier, Pool
from tqdm import tqdm


FEATURES_BATCH_SIZE = 500
# No for bg, ar, ja, fa
SPACY_DICT = {
    "fr": "fr_core_news_sm",
    "en": "en_core_web_sm",
    "pt": "pt_core_news_sm",
    "ca": "ca_core_news_sm",
    "es": "es_core_news_sm",
    "de": "de_core_news_sm",
    "it": "it_core_news_sm",
    "ru": "ru_core_news_sm",
    "el": "el_core_news_sm",
    "pl": "pl_core_news_sm",
    "uk": "uk_core_news_sm",
    "zh": "zh_core_web_sm",
    "ko": "ko_core_news_sm",
}
SPACY_DEFAULT_MODEL = "en"


class LFClassifier:
    """
    Baseline model based on Linguistic features + Catboost classifier
    """

    name = "Linguistic features classifier"
    allows_classification: bool = True

    def __init__(self, params: Dict = {}):
        self.model = None
        self.searched_features = [f['key'] for f in lftk.search_features(language="general")]
        self.spacy_pipelines = {lang: spacy.load(SPACY_DICT[lang]) for lang in tqdm(SPACY_DICT.keys())}

    def save(self, filename):
        # Use joblib to save the class method
        self.searched_features = []
        self.spacy_pipelines = {}
        joblib.dump(self, filename)

    @classmethod
    def load(cls, filename):
        # Use joblib to load the class method
        loaded_instance = joblib.load(filename)
        loaded_instance.searched_features = [f['key'] for f in lftk.search_features(language="general")]
        loaded_instance.spacy_pipelines = {lang: spacy.load(SPACY_DICT[lang]) for lang in tqdm(SPACY_DICT.keys())}
        return loaded_instance

    def fit(self, trainset: pd.DataFrame, **kwargs) -> None:
        """
        Method used for model training
        Not needed in the baseline model
        """
        train_features_dict = self._prepare_features(trainset)

        train_part = trainset[trainset.is_val == False]
        val_part = trainset[trainset.is_val == True]
        train_features = [train_features_dict[text] for text in train_part["generated_text"].tolist() + train_part["real_text"].tolist()]
        train_labels = [1] * len(train_part) + [0] * len(train_part)
        validation_features = [train_features_dict[text] for text in val_part["generated_text"].tolist() + val_part["real_text"].tolist()]
        validation_labels = [1] * len(val_part) + [0] * len(val_part)

        print("Number of training samples:", len(train_labels))
        print("Number of validation samples:", len(validation_labels))

        # Building classifier model:
        train_data = Pool(
            data=pd.DataFrame(train_features),
            label=train_labels,
            cat_features=["language"],
        )
        validation_data = Pool(
            data=pd.DataFrame(validation_features),
            label=validation_labels,
            cat_features=["language"],
        )
        self.model = CatBoostClassifier(
            iterations=5000, 
            metric_period=100, 
            verbose=True, 
            learning_rate=0.01,
            early_stopping_rounds=500,
            eval_metric='Accuracy'
        )
        self.model.fit(train_data, eval_set=validation_data, use_best_model=True, verbose=True, plot=False)

    def predict(self, testset: pd.DataFrame, **kwargs) -> pd.DataFrame:
        """
        Model used for prediction using trained model
        """
        # Calculate test features:
        test_features_dict = self._prepare_features(testset)
        test_features = [test_features_dict[text] for text in testset["generated_text"].tolist() + testset["real_text"].tolist()]
        # Calculate test scores:
        scores = self.model.predict_proba(
            [
                [features_sample[k] for k in self.model.feature_names_] 
                for features_sample in test_features]
        )[:, 1]

        scores_dict = {text: score for text, score in zip(
            testset["generated_text"].tolist() + testset["real_text"].tolist(),
            scores
        )}
        testset["generated_text_score"] = testset["generated_text"].apply(lambda x: scores_dict[x])
        testset["real_text_score"] = testset["real_text"].apply(lambda x: scores_dict[x])
        return testset

    def _prepare_features(self, dataset: pd.DataFrame):

        features = {}  # Key is a text, value is list of features
        texts = dataset["generated_text"].tolist() + dataset["real_text"].tolist()
        languages = dataset["generated_text_language"].apply(lambda x: x.lower()).tolist() + \
            dataset["real_text_language"].apply(lambda x: x.lower()).tolist()
        full_dataset = pd.DataFrame({"text": texts, "language": languages})
        full_dataset.drop_duplicates(subset=["text"], inplace=True)
        
        for lang in full_dataset["language"].unique():
            print(f"Processing language: {lang}...")
            lang_features = []
            lang_texts = full_dataset[full_dataset["language"] == lang]["text"].tolist()
            nlp = self.spacy_pipelines.get(lang, self.spacy_pipelines[SPACY_DEFAULT_MODEL])
            for i in tqdm(range(0, len(lang_texts), FEATURES_BATCH_SIZE)):
                docs = list(nlp.pipe(lang_texts[i:i + FEATURES_BATCH_SIZE]))
                lang_features += lftk.Extractor(docs=docs).extract(self.searched_features)
            # Add language as a feature:
            for f in lang_features:
                f["language"] = lang
            for i, text in enumerate(lang_texts):
                features[text] = lang_features[i]
        return features


In [ ]:
import joblib
from tqdm import tqdm
import pandas as pd

train = pd.read_parquet("../../../data/train_data.parquet")
print("Number of samples to process train:", len(train))
test = pd.read_parquet("../../../data/test_data.parquet")
print("Number of samples to process test:", len(test))

In [ ]:
classifier = LFClassifier()
classifier.fit(train)

In [ ]:
classifier.save("lb_classifier.mod")
# classifier = LFClassifier().load("lb_classifier.mod")

# Inference logic for test datasets: 

In [ ]:
classifier = LFClassifier().load("lb_classifier.mod")

In [ ]:
import joblib
from tqdm import tqdm
import pandas as pd

train = pd.read_parquet("../../../data/train_data.parquet")
print("Number of samples to process train:", len(train))
test = pd.read_parquet("../../../data/test_data.parquet")
print("Number of samples to process test:", len(test))

In [ ]:
prediction = classifier.predict(test)
val_prediction = classifier.predict(train[train.is_val==True])

In [ ]:
# prediction format: (model, pipeline, user_id, thread_id): (score_real, score_predicted)
test_scores, val_scores = {}, {}
for _, row in prediction.iterrows():
    test_scores[(row["model"], row["pipeline"], row["user_id"], row["thread_id"])] = \
        (row["real_text_score"], row["generated_text_score"])
for _, row in val_prediction.iterrows():
    val_scores[(row["model"], row["pipeline"], row["user_id"], row["thread_id"])] = \
        (row["real_text_score"], row["generated_text_score"])

joblib.dump(test_scores, "lfc_our_test_scores.data")
joblib.dump(val_scores, "lfc_our_val_scores.data")

# Predicting scores for external datasets: 

In [ ]:
multisocial = pd.read_csv("../../../data/external_data/multisocial/multisocial_anonymized.csv")
multisocial = multisocial[multisocial.split == "test"]

In [ ]:
def make_predictions(model: LFClassifier, list_of_texts: list[str], list_of_languages: list[str]) -> list:
    """
    Function to make predictions with LFClassifier model
    on a list of texts and their corresponding languages
    """
    dataset = pd.DataFrame({
        "generated_text": list_of_texts,
        "generated_text_language": list_of_languages,
        "real_text": ["placeholder text"] * len(list_of_texts),  # Dummy column
        "real_text_language": ["en"] * len(list_of_texts),  # Dummy column
    })
    predictions_df = model.predict(dataset)
    return predictions_df["generated_text_score"].tolist()

In [ ]:
multisocial["scores"] = make_predictions(classifier, multisocial["text"].to_list(), multisocial["language"].to_list())

In [ ]:
# prediction format: (id): (score)
ms_scores = {}
for i, row in multisocial.iterrows():
    ms_scores[i] = row["scores"]

joblib.dump(ms_scores, "lfc_ms_test_scores.data")

In [ ]:
AIGTBench = pd.read_parquet("../../../data/external_data/AIGTBench")
AIGTBench = AIGTBench[AIGTBench.social_media_platform == "reddit"]

In [ ]:
AIGTBench["scores"] = make_predictions(classifier, AIGTBench["text"].to_list(), ["en"] * len(AIGTBench))

In [ ]:
# prediction format: (id): (score)
AIGTBench_scores = {}
for i, row in AIGTBench.iterrows():
    AIGTBench_scores[i] = row["scores"]

joblib.dump(AIGTBench_scores, "lfc_AIGTBench_test_scores.data")

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(AIGTBench.label, AIGTBench.scores)
print("AIGTBench ROC AUC:", auc)

auc = roc_auc_score(multisocial.label, multisocial.scores)
print("multisocial ROC AUC:", auc)

In [ ]:
import json
from tqdm import tqdm
import pandas as pd
data_path = "../../../data/external_data/fox8_23_dataset.ndjson"
with open(data_path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

fox8_23_dicts = []
for sample in tqdm(data):
    label = 0 if sample["label"] == "human" else 1
    user_id = sample["user_id"]
    for tweet in sample["user_tweets"]:
        fox8_23_dicts.append({
            "text": tweet["text"],
            "user_id": user_id,
            "label": label
        })
fox8_23 = pd.DataFrame(fox8_23_dicts)
fox8_23["scores"] = make_predictions(classifier, fox8_23["text"].to_list(), ["en"] * len(fox8_23))

# prediction format: (id): (score)
fox8_23_scores = {}
for i, row in fox8_23.iterrows():
    fox8_23_scores[i] = row["scores"]

joblib.dump(fox8_23_scores, "lfc_fox8_23_scores.data")

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(fox8_23.label, fox8_23.scores)
print("fox8_23 ROC AUC:", auc)